In [15]:
%pip install -q fsspec huggingface-hub pandas pyarrow


[notice] A new release of pip is available: 25.3 -> 26.1.2
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [16]:
import pandas as pd
from src.constants import Column, DOWNLOAD_DIR, INPUT_DIR, OUTPUT_DIR, TEMP_DIR

# Gelabelte Trainingsdaten vorbereiten

In [17]:
splits = {
    "train": "data/train-00000-of-00001.parquet",
    "validation": "data/validation-00000-of-00001.parquet",
    "test": "data/test-00000-of-00001.parquet",
}

frames = []

for split_name, path in splits.items():
    split_df = pd.read_parquet("hf://datasets/Flowerly/modern-fake-reviews/" + path)
    split_df[Column.SPLIT] = split_name
    frames.append(split_df)

df = pd.concat(frames, ignore_index=True)

electronics_df = (
    df[df[Column.CATEGORY] == "Electronics_5"]
    .copy()
    .reset_index(drop=True)
)

electronics_df[Column.LABEL] = electronics_df[Column.LABEL].map({
    "OR": 0,
    "CG": 1,
})
electronics_df.insert(0, Column.ID, range(len(electronics_df)))

electronics_df.to_csv(INPUT_DIR / "electronics_raw.csv", index=False)

print(len(electronics_df))

3986


# PC-Monitor-Testdaten vorbereiten

In [18]:
meta_file = DOWNLOAD_DIR / "meta_Electronics.jsonl"
review_file = DOWNLOAD_DIR / "Electronics.jsonl"

In [19]:
monitor_meta_file = TEMP_DIR / "monitor_meta.csv"

def is_pc_monitor(categories):
    categories = categories or []
    return (
        "Computers & Accessories" in categories
        and "Monitors" in categories
    )


columns_needed = [
    "parent_asin",
    "title",
    "main_category",
    "average_rating",
    "rating_number",
    "price",
    "store",
    "categories",
    "details"
]

first_chunk = True

for chunk in pd.read_json(
    meta_file,
    lines=True,
    chunksize=100_000,
):
    # Nur benötigte Spalten behalten
    chunk = chunk[columns_needed]

    # Nur PC-Monitore behalten
    mask = chunk["categories"].apply(is_pc_monitor)
    chunk = chunk[mask]

    # Stückweise an CSV anhängen
    chunk.to_csv(
        monitor_meta_file,
        mode="w" if first_chunk else "a",
        header=first_chunk,
        index=False
    )

    first_chunk = False

In [20]:
pc_monitors = pd.read_csv(
    monitor_meta_file
)

exclude_terms = [
    "raspberry pi",
    "raspb pi",
    "rpi",
    "temperature",
    "temp monitor",
    "repair kit",
    "controller board",
    "arcade machine",
    "game hat",
    "lcd module",
    "car monitor",
    "gauge cluster",
    "fitness tracker",
    "heart rate",
    "screen protector"
]

title_lower = pc_monitors["title"].fillna("").str.lower()

exclude_mask = False

for term in exclude_terms:
    exclude_mask = (
        exclude_mask
        | title_lower.str.contains(term, regex=False)
    )

pc_monitors_clean = pc_monitors[~exclude_mask].copy()

total_ratings = pc_monitors_clean["rating_number"].sum()

print("Produkte:", len(pc_monitors_clean))
print("Summe rating_number:", total_ratings)

print(
    pc_monitors_clean["rating_number"].describe(
        percentiles=[0.25, 0.5, 0.75, 0.90, 0.95, 0.99]
    )
)

Produkte: 9292
Summe rating_number: 1610820
count     9292.000000
mean       173.355575
std       1020.378359
min          1.000000
25%          3.000000
50%         13.000000
75%         57.250000
90%        258.900000
95%        676.450000
99%       2931.090000
max      40868.000000
Name: rating_number, dtype: float64


In [21]:
monitor_asins = set(
    pc_monitors_clean["parent_asin"]
    .dropna()
    .unique()
)

monitor_reviews_file = TEMP_DIR / "monitor_reviews.csv"

chunk_number = 0
total_reviews = 0
first_chunk = True

for chunk in pd.read_json(
    review_file,
    lines=True,
    chunksize=100_000,
):
    chunk_number += 1

    # Nur Reviews von PC-Monitoren
    chunk = chunk[
        chunk["parent_asin"].isin(monitor_asins)
    ]

    total_reviews += len(chunk)

    if not chunk.empty:
        chunk.to_csv(
            monitor_reviews_file,
            mode="w" if first_chunk else "a",
            header=first_chunk,
            index=False
        )

        first_chunk = False

    if chunk_number % 10 == 0:
        print(
            f"Chunk {chunk_number} verarbeitet | "
            f"Gefundene Reviews: {total_reviews}"
        )

print("Fertig.")
print("Insgesamt gefundene Monitor-Reviews:", total_reviews)

Chunk 10 verarbeitet | Gefundene Reviews: 6566
Chunk 20 verarbeitet | Gefundene Reviews: 13390
Chunk 30 verarbeitet | Gefundene Reviews: 19854
Chunk 40 verarbeitet | Gefundene Reviews: 26821
Chunk 50 verarbeitet | Gefundene Reviews: 33213
Chunk 60 verarbeitet | Gefundene Reviews: 39723
Chunk 70 verarbeitet | Gefundene Reviews: 46562
Chunk 80 verarbeitet | Gefundene Reviews: 53251
Chunk 90 verarbeitet | Gefundene Reviews: 59690
Chunk 100 verarbeitet | Gefundene Reviews: 65896
Chunk 110 verarbeitet | Gefundene Reviews: 72094
Chunk 120 verarbeitet | Gefundene Reviews: 78547
Chunk 130 verarbeitet | Gefundene Reviews: 85625
Chunk 140 verarbeitet | Gefundene Reviews: 92203
Chunk 150 verarbeitet | Gefundene Reviews: 98770
Chunk 160 verarbeitet | Gefundene Reviews: 105759
Chunk 170 verarbeitet | Gefundene Reviews: 112496
Chunk 180 verarbeitet | Gefundene Reviews: 119280
Chunk 190 verarbeitet | Gefundene Reviews: 126071
Chunk 200 verarbeitet | Gefundene Reviews: 132845
Chunk 210 verarbeitet | G

In [24]:
monitor_reviews = pd.read_csv(monitor_reviews_file)
product_lookup = (
    pc_monitors_clean[["parent_asin", "title", "price"]]
    .drop_duplicates(subset="parent_asin")
    .rename(columns={"title": "product"})
)
monitor_reviews = monitor_reviews.merge(
    product_lookup,
    how="left",
    on="parent_asin",
    validate="many_to_one",
)

SAMPLE_SIZE = 5_000
RANDOM_STATE = 7

# Die ursprüngliche Review-Population bleibt erhalten: Es wird weder nach
# Sternen ausgeglichen noch die Anzahl der Reviews pro Produkt begrenzt.
sample_candidates = (
    monitor_reviews
    .dropna(subset=["parent_asin", "text", "rating"])
    .drop_duplicates(subset=["asin", "user_id", "timestamp"])
)

if len(sample_candidates) < SAMPLE_SIZE:
    raise ValueError(
        f"Es sind nur {len(sample_candidates)} eindeutige Reviews verfügbar; "
        f"für die Stichprobe werden {SAMPLE_SIZE} benötigt."
    )

monitor_reviews_sample_raw = sample_candidates.sample(
    n=SAMPLE_SIZE,
    random_state=RANDOM_STATE,
).reset_index(drop=True)

print("Natürliche Sterneverteilung der Stichprobe:")
print(
    monitor_reviews_sample_raw["rating"]
    .value_counts()
    .sort_index()
)
print(
    "Unterschiedliche Produkte:",
    monitor_reviews_sample_raw["parent_asin"].nunique(),
)

monitor_sample = pd.DataFrame({
    Column.CATEGORY: "PC monitors",
    Column.ASIN: monitor_reviews_sample_raw["asin"],
    Column.PRODUCT: monitor_reviews_sample_raw["product"],
    Column.PRICE: monitor_reviews_sample_raw["price"],
    Column.TITLE: monitor_reviews_sample_raw["title"],
    Column.TEXT: monitor_reviews_sample_raw["text"],
    Column.RATING: monitor_reviews_sample_raw["rating"],
})
monitor_sample.insert(0, Column.ID, range(len(monitor_sample)))

monitor_sample.to_csv(
    INPUT_DIR / "sample_raw.csv",
    index=False
)

Natürliche Sterneverteilung der Stichprobe:
rating
1     563
2     272
3     355
4     720
5    3090
Name: count, dtype: int64
Unterschiedliche Produkte: 1633
